# 时间序列分析 Time Series Analysis

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

时间序列分析涉及对按时间顺序排列的数据进行处理和预测。本notebook涵盖使用深度学习方法进行时间序列预测的基础知识。

Time series analysis involves processing and predicting data ordered in time. This notebook covers the fundamentals of time series forecasting using deep learning approaches.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/timeseries.png" width=500>

# 概述 Overview

* **目标:**  基于历史数据预测未来值。
* **优点:** 
  * 可捕捉序列中的时间依赖性
  * 支持多变量预测
  * 可处理长期依赖（LSTM/GRU）
* **缺点:**
  * 对噪声敏感
  * 需要大量数据
* **其他:** 
  * 广泛应用于金融、气象、医疗等领域
  * Transformer 也被用于时间序列

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# 数据生成 Generate Data

In [ ]:
# 生成正弦波数据 Generate sine wave data
def generate_sin_data(num_samples=1000, seq_length=20):
    X = []
    y = []
    for i in range(num_samples):
        start = np.random.randint(0, 2 * np.pi, 1)
        seq = np.sin(np.linspace(start, start + seq_length, seq_length + 1))
        X.append(seq[:seq_length])
        y.append(seq[seq_length])
    return np.array(X), np.array(y)

seq_length = 20
X, y = generate_sin_data(num_samples=1000, seq_length=seq_length)

# 划分训练集和测试集 Split train and test
train_size = int(len(X) * 0.8)
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

# 转换为张量 Convert to tensors
X_train = torch.FloatTensor(X_train).unsqueeze(-1)  # (N, seq_len, 1)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test).unsqueeze(-1)
y_test = torch.FloatTensor(y_test)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')

# LSTM 模型 LSTM Model

In [ ]:
class LSTMTimeSeries(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=1):
        super(LSTMTimeSeries, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM 层
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # 全连接层
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # LSTM 前向传播
        out, _ = self.lstm(x)
        # 只取最后一个时间步
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
# 初始化模型 Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMTimeSeries().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [ ]:
# 训练 Training
num_epochs = 50
batch_size = 32

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i+batch_size].to(device)
        batch_y = y_train[i:i+batch_size].to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_X).squeeze()
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}')

In [ ]:
# 预测 Prediction
model.eval()
with torch.no_grad():
    predictions = model(X_test.to(device)).cpu().squeeze()
    test_loss = criterion(predictions, y_test)
    print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# 可视化 Visualization
plt.figure(figsize=(12, 6))
plt.plot(y_test.numpy(), label='True Values')
plt.plot(predictions.numpy(), label='Predictions')
plt.legend()
plt.title('Time Series Prediction')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.show()

# TODO

- 多变量时间序列 Multivariate Time Series
- 注意力机制用于时间序列 Attention for Time Series
- Temporal Fusion Transformers
- 股票价格预测 Stock Price Prediction